In [ ]:

def ssfmDBP(Ei, param):
    """
    Split-step Fourier digital backpropagation (symmetric, single-pol.).

    Mirrors the structure of manakovDBP exactly, adapted for single polarization.
    Can be placed either:
      - On the optical field before coherent detection (amp='edfa' or 'ideal')
      - After matched filter, before decimation (amp='edfa' or 'ideal', with
        power renormalisation to launch power before calling)

    The linear operator always includes +α/2 to reverse distributed fibre loss
    within each step. The amp parameter only controls whether the lumped EDFA
    gain is reversed at the start of each span.

    Parameters
    ----------
    Ei : np.array
        Input signal field (optical or electrical complex baseband).
    param : optic.utils.parameters object
        - param.Ltotal  : total fiber length [km]           [default: 400]
        - param.Lspan   : span length [km]                  [default: 80]
        - param.hz      : fixed step size [km]              [default: 0.5]
        - param.alpha   : attenuation [dB/km]               [default: 0.2]
        - param.D       : dispersion [ps/nm/km]             [default: 16]
        - param.gamma   : nonlinear coeff. [1/W/km]         [default: 1.3]
        - param.Fc      : carrier frequency [Hz]            [default: 193.1e12]
        - param.Fs      : sampling frequency [Hz]           [required]
        - param.prec    : numerical precision               [default: np.complex128]
        - param.amp     : 'edfa', 'ideal', or None          [default: 'edfa']
        - param.prgsBar : show progress bar                 [default: True]
        - param.returnParameters : return param object      [default: False]

    Returns
    -------
    Ech : np.array
        Signal after digital backpropagation.

    References
    ----------
    [1] E. Ip and J. M. Kahn, "Compensation of Dispersion and Nonlinear
        Impairments Using Digital Backpropagation," JLT, vol. 26, no. 20,
        pp. 3416-3425, 2008, doi: 10.1109/JLT.2008.927791.
    [2] L. B. Du and A. J. Lowery, "Improved single channel backpropagation
        for intra-channel fiber nonlinearity compensation," Opt. Express,
        vol. 18, no. 16, pp. 17075-17088, 2010, doi: 10.1364/OE.18.017075.
    """
    try:
        Fs = param.Fs
    except AttributeError:
        logg.error("Simulation sampling frequency (Fs) not provided.")

    param.Ltotal           = getattr(param, "Ltotal",          400)
    param.Lspan            = getattr(param, "Lspan",            80)
    param.hz               = getattr(param, "hz",              0.5)
    param.alpha            = getattr(param, "alpha",           0.2)
    param.D                = getattr(param, "D",                16)
    param.gamma            = getattr(param, "gamma",           1.3)
    param.Fc               = getattr(param, "Fc",        193.1e12)
    param.prec             = getattr(param, "prec",   np.complex128)
    param.amp              = getattr(param, "amp",          "edfa")
    param.prgsBar          = getattr(param, "prgsBar",         True)
    param.returnParameters = getattr(param, "returnParameters", False)

    Ltotal           = param.Ltotal
    Lspan            = param.Lspan
    hz               = param.hz
    alpha            = param.alpha
    D                = param.D
    gamma            = param.gamma
    Fc               = param.Fc
    prec             = param.prec
    amp              = param.amp
    prgsBar          = param.prgsBar
    returnParameters = param.returnParameters

    c_kms = const.c / 1e3
    λ     = c_kms / Fc
    α     = alpha / (10 * np.log10(np.exp(1)))
    β2    = -(D * λ**2) / (2 * np.pi * c_kms)
    γ     = gamma

    Nspans = int(np.floor(Ltotal / Lspan))
    Nsteps = int(np.floor(Lspan / hz))

    Nfft = len(Ei)
    ω    = 2 * np.pi * Fs * fftfreq(Nfft).astype(prec)

    # ── Linear operator ──────────────────────────────────────────────────
    # Exact inverse of forward ssfm operator:
    #   forward:  exp(-(α/2)*(hz/2) + 1j*(β2/2)*ω²*(hz/2))
    #   inverse:  exp(+(α/2)*(hz/2) - 1j*(β2/2)*ω²*(hz/2))
    #
    # This matches manakovDBP: argLimOp = (α/2) - 1j*(β2/2)*ω²
    # The +α/2 term reverses distributed fibre loss within each step.
    # It is always present regardless of amp mode.
    # The amp parameter only controls the lumped EDFA gain reversal.
    argLimOp = np.array((α / 2) - 1j * (β2 / 2) * (ω**2)).astype(prec)
    linOperator = np.exp(argLimOp * (hz / 2))

    Ech = Ei.reshape(len(Ei),)

    for _ in tqdm(range(1, Nspans + 1), disable=not prgsBar):

        # ── Reverse lumped EDFA gain at start of each span ───────────────
        # Mirrors manakovDBP exactly:
        #   amp='edfa' or 'ideal' → reverse exp(+α/2 * Lspan) gain
        #   amp=None              → exp(0) = no gain reversal
        if amp in {"edfa", "ideal"}:
            Ech = Ech * np.exp(-α / 2 * Lspan)
        elif amp is None:
            Ech = Ech * np.exp(0)

        # ── Reverse fiber steps ──────────────────────────────────────────
        for _ in range(1, Nsteps + 1):

            # First reversed linear half-step (frequency domain)
            Ech = ifft(fft(Ech) * linOperator)

            # Reversed nonlinear step — minus sign inverts Kerr phase
            # forward: exp(+1j*γ*|E|²*hz)
            # inverse: exp(-1j*γ*|E|²*hz)
            Ech = Ech * np.exp(-1j * γ * (Ech * np.conj(Ech)) * hz)

            # Second reversed linear half-step (frequency domain)
            Ech = ifft(fft(Ech) * linOperator)

    return (Ech.reshape(len(Ech),), param) if returnParameters else Ech.reshape(len(Ech),)

In [ ]:
def pconst(x, lim=True, R=1.25, pType="fancy", cmap="turbo", whiteb=True, label_size=17):
    """
    Plot signal constellations.
    (docstring unchanged except for label_size already added)
    """
    if type(x) == list:
        for ind, _ in enumerate(x):
            x[ind] = pnorm(x[ind])
        try:
            x[0].shape[1]
        except IndexError:
            x[0] = x[0].reshape(len(x[0]), 1)

        nSubPts = x[0].shape[1]
        radius = R * np.sqrt(signalPower(x[0]))
    else:
        x = pnorm(x)
        try:
            x.shape[1]
        except IndexError:
            x = x.reshape(len(x), 1)

        nSubPts = x.shape[1]
        radius = R * np.sqrt(signalPower(x))

    if nSubPts > 1:
        if nSubPts < 5:
            nCols = nSubPts
            nRows = 1
        elif nSubPts >= 6:
            nCols = int(np.ceil(nSubPts / 2))
            nRows = 2

        Position = range(1, nSubPts + 1)

        fig = plt.figure()

        if type(x) == list:
            for k in range(nSubPts):
                for ind in range(len(x)):
                    if pType == "fancy":
                        if ind == 0:
                            ax = fig.add_subplot(
                                nRows, nCols, Position[k], projection="scatter_density"
                            )
                        ax = constHist(x[ind][:, k], ax, cmap, whiteb)
                    elif pType == "fast":
                        if ind == 0:
                            ax = fig.add_subplot(nRows, nCols, Position[k])
                        ax.plot(x[ind][:, k].real, x[ind][:, k].imag, ".")

                ax.axis("square")
                ax.set_xlabel("In-Phase (I)", fontsize=label_size)
                ax.set_ylabel("Quadrature (Q)", fontsize=label_size)
                ax.tick_params(axis='both', labelsize=label_size)  # <-- added
                ax.set_title(f"mode {str(Position[k] - 1)}")

                if lim:
                    ax.set_xlim(-radius, radius)
                    ax.set_ylim(-radius, radius)
        else:
            for k in range(nSubPts):
                if pType == "fancy":
                    ax = fig.add_subplot(
                        nRows, nCols, Position[k], projection="scatter_density"
                    )
                    ax = constHist(x[:, k], ax, cmap, whiteb)
                elif pType == "fast":
                    ax = fig.add_subplot(nRows, nCols, Position[k])
                    ax.plot(x[:, k].real, x[:, k].imag, ".")

                ax.axis("square")
                ax.set_xlabel("In-Phase (I)", fontsize=label_size)
                ax.set_ylabel("Quadrature (Q)", fontsize=label_size)
                ax.tick_params(axis='both', labelsize=label_size)  # <-- added
                ax.set_title(f"mode {str(Position[k] - 1)}")

                if lim:
                    ax.set_xlim(-radius, radius)
                    ax.set_ylim(-radius, radius)

        fig.tight_layout()

    elif nSubPts == 1:
        fig = plt.figure()
        if pType == "fancy":
            ax = fig.add_subplot(1, 1, 1, projection="scatter_density")
            ax = constHist(x[:, 0], ax, cmap, whiteb)
        elif pType == "fast":
            ax = plt.gca()
            ax.plot(x.real, x.imag, ".")
        plt.axis("square")
        ax.set_xlabel("In-Phase (I)", fontsize=label_size)
        ax.set_ylabel("Quadrature (Q)", fontsize=label_size)
        ax.tick_params(axis='both', labelsize=label_size)  # <-- added

        if lim:
            plt.xlim(-radius, radius)
            plt.ylim(-radius, radius)

    #plt.show()
    #plt.pause(0.01)

    return fig, ax